In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
# Load adjacency matrix (34x34) and faction labels (34,)
A = np.loadtxt('../datasets/karate_adjacency.csv', delimiter=',')
F = np.loadtxt('../datasets/karate_factions.csv', delimiter=',')

print('Adjacency matrix shape:', A.shape)
print('Faction labels shape:', F.shape)
print('Unique faction values:', np.unique(F))

In [ ]:
# Build NetworkX graph from adjacency matrix
G = nx.from_numpy_array(A)

# Color nodes by ground-truth faction (1 -> blue, 2 -> red)
faction_colors = ['steelblue' if f == 1.0 else 'tomato' for f in F]

pos = nx.spring_layout(G, seed=42)

fig, ax = plt.subplots(figsize=(8, 6))
nx.draw_networkx(G, pos=pos, node_color=faction_colors,
                 with_labels=True, node_size=300,
                 font_size=8, ax=ax)
ax.set_title("Zachary's karate club — ground-truth factions\n(blue = faction 1, red = faction 2)")
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Modularity matrix  B = A - d*d^T / (2m)
D = A.sum(axis=1)          # degree vector
m = A.sum() / 2            # number of edges
B = A - np.outer(D, D) / (2 * m)

print(f'Number of edges m = {m:.0f}')
print(f'Modularity matrix B shape: {B.shape}')
print(f'Row sums of B (should all be ~0): {B.sum(axis=1).round(8)}')

In [ ]:
# Eigendecomposition of B — sort by descending eigenvalue
w, v = np.linalg.eig(B)
w = w.real
v = v.real

sort_idx = np.argsort(w)[::-1]
w_sorted = w[sort_idx]
v_sorted = v[:, sort_idx]

print('Top 5 eigenvalues:', w_sorted[:5].round(4))

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(len(w_sorted)), w_sorted)
ax.axhline(0, color='red', linewidth=0.8)
ax.set_xlabel('Eigenvalue index (sorted descending)')
ax.set_ylabel('Eigenvalue')
ax.set_title('Spectrum of the modularity matrix B')
plt.tight_layout()
plt.show()

In [ ]:
# Community assignment: sign of the leading eigenvector
leading_eigvec = v_sorted[:, 0]
community_pred = np.where(leading_eigvec >= 0, 1, 2).astype(float)

print('Predicted community labels:', community_pred.astype(int))
print('Counts:', dict(zip(*np.unique(community_pred, return_counts=True))))

In [ ]:
# Modularity Q using ground-truth faction vector S in {-1, +1}
S = 2 * (F - 1) - 1   # maps 1 -> -1, 2 -> +1
Q = float(S @ B @ S) / (4 * m)
print(f'Modularity Q (ground-truth factions): {Q:.6f}')

In [ ]:
# Compare predicted communities with ground-truth factions
# Handle possible label flip
match_direct  = np.mean(community_pred == F)
match_flipped = np.mean((3 - community_pred) == F)   # flip 1<->2
accuracy = max(match_direct, match_flipped)

print(f'Agreement (direct):  {match_direct:.4f}')
print(f'Agreement (flipped): {match_flipped:.4f}')
print(f'Best agreement:      {accuracy:.4f}')

# Align labels for display
if match_flipped > match_direct:
    community_display = 3 - community_pred
else:
    community_display = community_pred

comparison = pd.DataFrame({
    'node': range(len(F)),
    'ground_truth': F.astype(int),
    'predicted':    community_display.astype(int),
    'correct':      (community_display == F)
})
display(comparison)

In [ ]:
# Plot graph colored by predicted communities
pred_colors = ['steelblue' if c == 1 else 'tomato' for c in community_display]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

nx.draw_networkx(G, pos=pos, node_color=faction_colors,
                 with_labels=True, node_size=300,
                 font_size=8, ax=axes[0])
axes[0].set_title('Ground-truth factions')
axes[0].axis('off')

nx.draw_networkx(G, pos=pos, node_color=pred_colors,
                 with_labels=True, node_size=300,
                 font_size=8, ax=axes[1])
axes[1].set_title(f'Predicted communities (spectral, Q={Q:.3f})')
axes[1].axis('off')

plt.suptitle("Zachary's karate club — community detection via modularity matrix",
             fontsize=12)
plt.tight_layout()
plt.show()

## Your answers here
